In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount = True)

# Change working directory
os.chdir("/content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度")
print("Current working directory:", os.getcwd())

Mounted at /content/drive
Current working directory: /content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度


In [2]:
def add_exchange_suffix(code: str) -> str:
    code = str(code)

    if len(code) == 5 and code.isdigit():
        ticker = f"{code}.HK"
        return ticker

    if code.startswith(("60", "68", "900")):
        ticker = f"{code}.SH"
        return ticker

    if code.startswith(("00", "30", "200")):
        ticker = f"{code}.SZ"
        return ticker

    return code

In [6]:
date_ranges = [
    (pd.Timestamp("2023-01-01"), pd.Timestamp("2023-07-01")),
    # (pd.Timestamp("2022-04-01"), pd.Timestamp("2022-05-01")),
    # (pd.Timestamp("2022-01-01"), pd.Timestamp("2022-04-01")),
]

def is_within_date_ranges(date, date_ranges):
    if pd.isna(date):
        return False
    for start, end in date_ranges:
        if start <= date < end:
            return True
    return False

In [9]:
translation_path = "equity_reports_en.csv"
logit_path       = "sentiment_logit.csv"
softmax_path     = "sentiment_softmax.csv"

processed_ids = set()

# --- Translation file setup ---
if os.path.exists(translation_path):
    df_existing = pd.read_csv(translation_path, usecols=['REPORT_ID'])
    processed_ids = set(df_existing['REPORT_ID'])
    print(f"Translation progress found: {len(processed_ids)} finished")
else:
    pd.DataFrame(columns=["REPORT_ID", "TITLE_EN", "SUMMARY_EN"]).to_csv(translation_path, index=False)
    print("Translation file created")

# --- Logit file setup ---
if os.path.exists(logit_path):
    df_existing = pd.read_csv(logit_path, usecols=['REPORT_ID'])
    processed_ids |= set(df_existing['REPORT_ID'])
    print(f"Logit progress found: {len(processed_ids)} finished")
else:
    pd.DataFrame(columns=[
        "REPORT_ID",
        "title_neg_logit", "title_neu_logit", "title_pos_logit",
        "summary_neg_logit", "summary_neu_logit", "summary_pos_logit"
    ]).to_csv(logit_path, index=False)
    print("Logit file created")

# --- Softmax file setup ---
if os.path.exists(softmax_path):
    df_existing = pd.read_csv(softmax_path, usecols=['REPORT_ID'])
    processed_ids |= set(df_existing['REPORT_ID'])
    print(f"Softmax progress found: {len(processed_ids)} finished")
else:
    pd.DataFrame(columns=[
        "REPORT_ID",
        "title_neg_prob", "title_neu_prob", "title_pos_prob",
        "summary_neg_prob", "summary_neu_prob", "summary_pos_prob"
    ]).to_csv(softmax_path, index=False)
    print("Softmax file created")

print(f"Total processed IDs tracked: {len(processed_ids)}")

Translation progress found: 500 finished
Logit progress found: 500 finished
Softmax progress found: 500 finished
Total processed IDs tracked: 500


In [5]:
equity_reports = pd.read_csv(
    'equity_reports.csv',
    encoding='utf-8',
    parse_dates=['PUBLISH_DATE', 'UPDATE_TIME', 'UPDATE_DATE'],
    dtype={"STOCK_CODE": str}
)

equity_reports.head()

,REPORT_ID,SUMMARY,TITLE,PUBLISH_DATE,STOCK_CODE,STOCK_ABBR,UPDATE_TIME,UPDATE_DATE,Lag
0,4408114,投资要点事件：近日，（1）国家药监局批准公司的1类创新药奥布替尼（商品名：宜诺凯）上...,诺诚健华-B（09969）：奥布替尼两项淋巴瘤适应症获批，纳入港股通期待估值修复,2020-12-29,09969,诺诚健华-B,2021-05-19 17:39:59,2021-05-19,141
1,4408146,战略明确，长期成长路径清晰，维持“买入”评级无论是中端期，还是长期视角看，公司均表现出优异的...,金山办公（688111）公司深度报告之二：循成长之迹，探WPS发展远景,2020-12-29,688111,金山办公,2020-12-29 09:32:28,2020-12-29,0
2,4408165,股权激励划定明确增长目标，有望充分激发公司市值与成长动力。公司发布2021年限制性股票激励计...,共创草坪（605099）：股权激励划定明确目标，人造草坪龙头再添增长动力,2020-12-29,605099,共创草坪,2021-03-09 12:15:20,2021-03-09,70
3,4408166,事件：公司拟收购合肥龙翔高复学校70%股权。公司于2020年12月28日与毛成红、合肥龙翔高...,科德教育（300192）：拟并购合肥龙翔高复学校，坚定推进教育产业布局,2020-12-29,300192,科德教育,2021-03-09 12:15:21,2021-03-09,70
4,4408220,NaN,华住集团-S（01179）跟踪报告：流量品牌技术三位一体，引领行业发展,2020-12-29,01179,华住集团-S,2021-02-03 16:50:49,2021-02-03,36


In [7]:
filtered_df = equity_reports[
    equity_reports['UPDATE_DATE'].apply(lambda d: is_within_date_ranges(d, date_ranges))
    & ~equity_reports['REPORT_ID'].isin(processed_ids)
    & (equity_reports['Lag'] <= 90)
    & equity_reports['STOCK_CODE'].str.startswith(('60','68','900','00','30','200'), na=False)
    & (equity_reports['STOCK_CODE'].str.len() == 6)
].reset_index(drop=True)

print(len(filtered_df))

32624


In [ ]:
filtered_df.to_csv("equity_reports_1.csv", index=False, encoding="utf-8-sig")

In [8]:
n = len(filtered_df)
chunk_size = n // 3

df1 = filtered_df.iloc[:chunk_size]
df2 = filtered_df.iloc[chunk_size:2*chunk_size]
df3 = filtered_df.iloc[2*chunk_size:]

df1.to_csv("equity_reports_1.csv", index=False, encoding="utf-8-sig")
df2.to_csv("equity_reports_2.csv", index=False, encoding="utf-8-sig")
df3.to_csv("equity_reports_3.csv", index=False, encoding="utf-8-sig")

In [ ]:
count = 0
for i, row in df1.iterrows():
    code = row['STOCK_CODE']
    if len(str(code)) == 6 and code.startswith(("60", "68", "900", "00", "30", "200")):
        count += 1

count

8838

In [ ]:
df1['STOCK_CODE'] = df1['STOCK_CODE'].apply(add_exchange_suffix)

/var/folders/wc/s3v7045n4fz9_4bmb56bprdw0000gp/T/ipykernel_10209/2717666451.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['STOCK_CODE'] = df1['STOCK_CODE'].apply(add_exchange_suffix)


In [ ]:
count = 0
for i, row in df1.iterrows():
    if i > 143: break

    ID = row['REPORT_ID']
    if ID in processed_ids:
        print("Skipped")
        continue

    ticker = str(row['STOCK_CODE'])

    if not (ticker.endswith('.SH') or ticker.endswith('.SZ')):
        print("Skipped")

    count += 1


count

Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped
Skipped


144